<a href="https://colab.research.google.com/github/rgomesa2025/MVP_MACHINE_LEARNING_-_ANALYTICS/blob/main/notebook/Producao_Nacional_Petroleo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVP — Machine Learning & Analytics

**PUC-RIO** - Pós-Graduação em Ciência de Dados e Analytics

**Nome:** _Rosângela Gomes André_

**Matrícula:** _4052025002132_

**Data:** _05/07/2026_

**Dataset:** _[Produção de Petróleo e gás natural](https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-estatisticos), após acesse [Produção por Poços](https://cdp.anp.gov.br/ords/r/cdp_apex/consulta-dados-publicos-cdp/consulta-produ%C3%A7%C3%A3o-por-po%C3%A7o)_

**Tipo de problema:** _Séries Temporais_

# 1. Definição do Problema

## 1.1 Contexto do Problema
A indústria de petróleo e gás gera um volume massivo de dados sobre a extração de recursos. O acesso a essas informações de forma pública permite analisar o histórico de produção dos principais campos nacionais, mas olhar apenas para o passado não basta para o planejamento de longo prazo. O desafio está em entender o comportamento futuro da produção diante da oscilação natural dos reservatórios.

### Previsão que o Modelo Deve Apoiar
O modelo de Machine Learning deve apoiar a **previsão de comportamento para os próximos 5 anos** de cinco variáveis fundamentais e interdependentes do dataset público:
1. Óleo
2. Óleo Condensado
3. Gás Natural
4. Gás Natural Associado
5. Gás Natural Não Associado

A solução vai gerar projeções volumétricas futuras para cada uma dessas curvas simultaneamente.

### Relevância do Problema
Prever a produção de hidrocarbonetos com um horizonte de 5 anos é crucial porque as decisões nesse setor envolvem investimentos de altíssimo valor e longo tempo de maturação. Uma previsão imprecisa de gás (associado ou não associado), por exemplo, pode resultar em falta de capacidade de escoamento ou gasodutos ociosos. Antecipar essas curvas reduz incertezas financeiras, otimiza a logística de transporte e garante maior segurança operacional e estratégica para o mercado de energia.

## 1.2 Objetivo do MVP

O objetivo deste MVP é desenvolver e avaliar modelos de Machine Learning capazes de projetar as curvas de produção de longo prazo (5 anos) para óleo, óleo condensado, gás natural, gás associado e gás não associado a partir do histórico de dados públicos do setor. A solução visa comparar o desempenho de algoritmos preditivos contra uma abordagem de referência (baseline), analisando a viabilidade técnica das projeções e mapeando as limitações operacionais do modelo.

## 1.3 Tipo de problema

* **Tipo escolhido:** Séries temporais / forecasting (combinado com Regressão).

* **Justificativa:** O projeto consiste na previsão de valores numéricos contínuos (volumes de produção de óleo e gás) onde a ordem cronológica dos dados é fundamental para o aprendizado do modelo. Não se trata de uma regressão simples ou classificação, pois o comportamento futuro das curvas depende diretamente do histórico temporal anterior. Por se tratar de um problema temporal, o ordenamento dos dados será estritamente respeitado na separação de treino e teste, evitando técnicas tradicionais de embaralhamento (shuffle) que causariam vazamento de dados do futuro para o passado.

## 1.4 Premissas, hipóteses e critérios de sucesso

**Hipóteses iniciais:**
1. A produção histórica recente e o comportamento de declínio natural dos reservatórios contêm padrões sazonais e de tendência suficientes para que algoritmos de Machine Learning projetem o comportamento futuro em um horizonte de 5 anos.
2. Existe uma correlação e interdependência significativa entre as 5 curvas (especialmente entre o óleo e o gás associado), permitindo que o modelo capture dinâmicas conjuntas de produção e entregue previsões mais consistentes do que projeções isoladas.
3. Modelos avançados de séries temporais ou de regressão baseados em árvores (como XGBoost ou LightGBM adaptados para contexto temporal) apresentarão um desempenho superior a uma abordagem estatística simples (baseline) para previsões de longo prazo.

**Critérios de sucesso:**
- **Métrica principal:** MAPE (Erro Percentual Absoluto Médio) e RMSE (Raiz do Erro Quadrático Médio). O MAPE será a métrica de negócio para avaliar o percentual médio de erro em cada uma das 5 curvas.
- **Resultado mínimo esperado:** Reduzir o MAPE em pelo menos 15% em comparação com o modelo baseline (que será definido pela repetição do último valor observado ou por uma média móvel simples).
- **Restrição prática:** Simplicidade e custo computacional. Como se trata de um MVP para previsão de longo prazo (estratégica, não em tempo real), o foco principal será a interpretabilidade das variáveis e a estabilidade do modelo ao longo dos 5 anos projetados, garantindo que o pipeline possa rodar sem a necessidade de infraestrutura de nuvem de altíssimo custo.

# 2. Ambiente, bibliotecas e reprodutibilidade

Esta seção reúne o ecossistema de software, bibliotecas e as configurações de semente (*seed*) necessárias para garantir que a execução deste notebook produza exatamente os mesmos resultados em qualquer ambiente compatível.

### Bibliotecas Utilizadas
O pipeline foi desenvolvido utilizando a linguagem Python, apoiando-se nos seguintes pacotes:
* **Manipulação e Análise de Dados:** `pandas` (para engenharia de recursos e estruturação das séries temporais) e `numpy` (para operações matemáticas de suporte).
* **Visualização de Dados:** `matplotlib` e `seaborn` (fundamentais para a análise exploratória das curvas de declínio e plotagem das projeções de 5 anos).
* **Modelagem Preditiva e Machine Learning:** `scikit-learn` (para divisão temporal dos dados, pré-processamento e cálculo de métricas como RMSE) e `xgboost` / `lightgbm` (algoritmos candidatos para a regressão multivariada adaptada ao contexto temporal).

### Reprodutibilidade (Seed Fixa)
Para garantir que as divisões de validação e a inicialização de determinados algoritmos não variem entre execuções, uma semente aleatória global foi definida.



In [3]:
#python
import os
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Validação e Divisão de Dados Temporais
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Modelos (Baseline e Candidatos Avançados)
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Métricas de Avaliação para Regressão
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import randint

warnings.filterwarnings("ignore")

# Configuração de semente fixa para reprodutibilidade
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("Python:", sys.version.split()[0])
print("Seed:", SEED)

Python: 3.12.13
Seed: 42


## 2.1 Dependências adicionais

Para a execução deste MVP, é necessária a instalação de pacotes complementares ao ecossistema padrão do Python. Embora as bibliotecas de manipulação de dados e regressão linear básica (como Pandas e Scikit-Learn) já sejam nativas em ambientes de ciência de dados, a modelagem preditiva das 5 curvas de hidrocarbonetos exige algoritmos avançados de Boosting.

Portanto, o projeto faz uso das seguintes dependências externas para garantir o treinamento dos modelos candidatos e a qualidade visual dos gráficos de projeção:



In [4]:
#python
# Executar a linha abaixo caso o ambiente não possua os pacotes instalados:
!pip install -q xgboost lightgbm seaborn

## 2.2 Funções auxiliares

Para garantir a organização do pipeline e evitar a repetição de código durante a fase de modelagem, foram desenvolvidas funções específicas para a avaliação do desempenho preditivo das curvas. Como o MVP lida com um problema de regressão temporal multivariada, as funções calculam os erros volumétricos com base nas quatro principais métricas de aderência do mercado de energia: MAE, RMSE, R² e o MAPE (que quantifica o percentual médio de erro diretamente no negócio).

O bloco abaixo centraliza essas funções para que possam ser chamadas de forma limpa na validação do modelo baseline e dos modelos candidatos:


In [6]:

#python
def calculate_mape(y_true, y_pred):
    """
    Calcula o Erro Percentual Absoluto Médio (MAPE).
    Remove valores nulos ou zerados do histórico para evitar divisões por zero.
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    if not np.any(mask):
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def evaluate_regression(y_true, y_pred):
    """
    Centraliza o cálculo das métricas de sucesso estabelecidas para as 5 curvas
    (Óleo, Condensado, Gás Natural, Gás Associado e Não Associado).
    """
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mse),
        "MAPE (%)": calculate_mape(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }


def show_results_table(results_dict):
    """
    Consolida as métricas dos experimentos em um DataFrame do Pandas,
    permitindo a comparação direta entre o modelo baseline e os algoritmos de Machine Learning.
    """
    return pd.DataFrame(results_dict).T